# 03 — Incremental Load Validation

This notebook validates the operational incremental flow and the
downstream Gold Star Schema.

Scope:

- inspect local Bronze state;
- compare local state with FIPEX releases;
- identify missing periods;
- validate catch-up behavior;
- inspect Silver coverage;
- build/validate the Gold dimensional model;
- validate the DuckDB analytical catalog.


In [ ]:
from pathlib import Path

import pandas as pd

from fipe_pipeline.duckdb_layer import (
    build_duckdb_catalog,
    connect_duckdb,
)
from fipe_pipeline.extract import (
    extract_missing_months,
    find_missing_periods,
    inspect_local_bronze,
    list_available_releases,
    validate_no_missing_remote_gap,
)
from fipe_pipeline.gold import build_gold

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_ROOT

## 1. Inspect local Bronze coverage


In [ ]:
inventory = inspect_local_bronze()
inventory

## 2. Inspect available FIPEX releases


In [ ]:
releases = list_available_releases()

[(release.period.label, release.tag) for release in releases[-10:]]

## 3. Identify missing periods


In [ ]:
missing = find_missing_periods(
    releases,
    inventory,
)

[(release.period.label, release.tag) for release in missing]

## 4. Validate remote continuity


In [ ]:
validate_no_missing_remote_gap(
    inventory,
    releases,
)

## 5. Execute catch-up idempotently


In [ ]:
catchup = extract_missing_months()
catchup

If the local Bronze layer is already current, `extraction_results`
should be empty. If one or more published months are missing, only
those months should be extracted.


## 6. Inspect Silver inventory


In [ ]:
silver_files = sorted(
    (PROJECT_ROOT / "data" / "silver").glob("year=*/month=*/fipe.parquet")
)

silver_inventory = []

for path in silver_files:
    period = pd.read_parquet(
        path,
        columns=[
            "ano_referencia",
            "mes_referencia",
        ],
    )

    unique_period = period.drop_duplicates()

    if len(unique_period) != 1:
        raise ValueError(
            f"Unexpected period cardinality in {path}: {len(unique_period)}"
        )

    row = unique_period.iloc[0]

    silver_inventory.append(
        {
            "year": int(row["ano_referencia"]),
            "month": int(row["mes_referencia"]),
            "rows": len(pd.read_parquet(path)),
            "path": str(path),
        }
    )

silver_inventory = pd.DataFrame(silver_inventory)

silver_inventory.tail()

In [ ]:
silver_inventory[["year", "month"]].iloc[[0, -1]]

In [ ]:
silver_inventory["rows"].sum()

In [ ]:
silver_inventory.duplicated(subset=["year", "month"]).sum()

## 7. Build the Gold Star Schema

Gold now consists of:

- `dim_date.parquet`
- `dim_vehicle.parquet`
- `fct_fipe_prices.parquet`

`vehicle_key` is a numeric surrogate key generated deterministically
from the sorted vehicle natural key.


In [ ]:
gold_result = build_gold(
    overwrite=True,
)

gold_result

## 8. Refresh and validate DuckDB


In [ ]:
duckdb_result = build_duckdb_catalog()
duckdb_result

In [ ]:
con = connect_duckdb()

con.sql("""
SELECT
    table_name,
    table_type
FROM information_schema.tables
WHERE table_name IN (
    'dim_date',
    'dim_vehicle',
    'fct_fipe_prices',
    'silver_fipe'
)
ORDER BY table_name
""").df()

## 9. Validate dimension/fact cardinalities


In [ ]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM dim_date) AS date_rows,
    (SELECT COUNT(*) FROM dim_vehicle) AS vehicle_rows,
    (SELECT COUNT(*) FROM fct_fipe_prices) AS fact_rows
""").df()

## 10. Validate foreign keys


In [ ]:
con.sql("""
SELECT
    SUM(
        CASE
            WHEN d.date_key IS NULL THEN 1
            ELSE 0
        END
    ) AS orphan_date_keys,
    SUM(
        CASE
            WHEN v.vehicle_key IS NULL THEN 1
            ELSE 0
        END
    ) AS orphan_vehicle_keys
FROM fct_fipe_prices AS f
LEFT JOIN dim_date AS d
    ON f.date_key = d.date_key
LEFT JOIN dim_vehicle AS v
    ON f.vehicle_key = v.vehicle_key
""").df()

## 11. Inspect the Star Schema


In [ ]:
con.sql("""
SELECT *
FROM dim_date
ORDER BY date_key DESC
LIMIT 5
""").df()

In [ ]:
con.sql("""
SELECT *
FROM dim_vehicle
ORDER BY vehicle_key
LIMIT 5
""").df()

In [ ]:
con.sql("""
SELECT *
FROM fct_fipe_prices
ORDER BY date_key DESC, vehicle_key
LIMIT 5
""").df()

In [ ]:
con.close()